In [ ]:
import pandas as pd

# Load TSV files
train_df = pd.read_csv("train.tsv", sep='\t', header=None)
val_df = pd.read_csv("valid.tsv", sep='\t', header=None)
test_df = pd.read_csv("test.tsv", sep='\t', header=None)

# Keep only statement and label
def preprocess(df):
    df = df[[1, 2]]  # Column 1: label, Column 2: statement
    df.columns = ['label', 'text']
    # Map labels to binary: 0 (Fake), 1 (Real)
    label_map = {
        'pants-fire': 0,
        'false': 0,
        'barely-true': 0,
        'half-true': 1,
        'mostly-true': 1,
        'true': 1
    }
    df['label'] = df['label'].map(label_map)
    df = df.dropna()  # Drop rows where label mapping failed
    return df

train_df = preprocess(train_df)
val_df = preprocess(val_df)
test_df = preprocess(test_df)

print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))


Train samples: 10240
Validation samples: 1284
Test samples: 1267


<ipython-input-1-bef0fbaed9c4>:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['label'] = df['label'].map(label_map)
<ipython-input-1-bef0fbaed9c4>:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['label'] = df['label'].map(label_map)
<ipython-input-1-bef0fbaed9c4>:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/u

# **Formatting a dataset so that it is compatible to be used with either PyTorch or Hugging face framework directly.**


In [ ]:
from transformers import BertTokenizer
from torch.utils.data import Dataset

# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Custom Dataset class
class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding=True, max_length=max_length)
        self.labels = labels.tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

# Create dataset objects
train_dataset = FakeNewsDataset(train_df['text'], train_df['label'], tokenizer)
val_dataset = FakeNewsDataset(val_df['text'], val_df['label'], tokenizer)
test_dataset = FakeNewsDataset(test_df['text'], test_df['label'], tokenizer)

print("Tokenization and dataset ready.")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Tokenization and dataset ready.


In [ ]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
import torch

# 👇 Add this custom trainer BEFORE the 'trainer = ...' block
class LoggingTrainer(Trainer):
    def training_step(self, model, inputs, num_items=None):
        # Standard training step from Hugging Face
        loss = super().training_step(model, inputs, num_items)

        # Extract loss value
        loss_value = loss.item() if hasattr(loss, 'item') else loss

        # Get current training state
        current_epoch = int(self.state.epoch) if self.state.epoch is not None else "?"
        current_step = self.state.global_step
        batch_size = self.args.per_device_train_batch_size
        batch_num = current_step % (len(self.train_dataset) // batch_size + 1)

        # Logging loss with epoch + batch
        print(f"Epoch {current_epoch} | Batch {batch_num} | Loss: {loss_value:.4f}")

        return loss

# Load pre-trained BERT for binary classification
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
print(f"Using model: {model.name_or_path}")

# Training configuration
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    evaluation_strategy="epoch",
    logging_dir='./logs',
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

# Define evaluation metric
from sklearn.metrics import accuracy_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

# ✅ Swap in LoggingTrainer instead of Trainer
trainer = LoggingTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# 🚀 Start training!
trainer.train()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Using model: bert-base-uncased
Epoch 0 | Batch 0 | Loss: 0.7742


Epoch,Training Loss,Validation Loss,Accuracy
1,0.648800,0.689911,0.584112
2,0.556900,0.699873,0.638629
3,0.274300,0.967503,0.635514


Epoch 0 | Batch 1 | Loss: 0.6604
Epoch 0 | Batch 2 | Loss: 0.7722
Epoch 0 | Batch 3 | Loss: 0.5853
Epoch 0 | Batch 4 | Loss: 0.6659
Epoch 0 | Batch 5 | Loss: 0.6298
Epoch 0 | Batch 6 | Loss: 0.7309
Epoch 0 | Batch 7 | Loss: 0.7224
Epoch 0 | Batch 8 | Loss: 0.7511
Epoch 0 | Batch 9 | Loss: 0.6909
Epoch 0 | Batch 10 | Loss: 0.6530
Epoch 0 | Batch 11 | Loss: 0.6974
Epoch 0 | Batch 12 | Loss: 0.6713
Epoch 0 | Batch 13 | Loss: 0.7424
Epoch 0 | Batch 14 | Loss: 0.6661
Epoch 0 | Batch 15 | Loss: 0.6737
Epoch 0 | Batch 16 | Loss: 0.7030
Epoch 0 | Batch 17 | Loss: 0.6931
Epoch 0 | Batch 18 | Loss: 0.7106
Epoch 0 | Batch 19 | Loss: 0.6729
Epoch 0 | Batch 20 | Loss: 0.6808
Epoch 0 | Batch 21 | Loss: 0.6970
Epoch 0 | Batch 22 | Loss: 0.7204
Epoch 0 | Batch 23 | Loss: 0.7423
Epoch 0 | Batch 24 | Loss: 0.6956
Epoch 0 | Batch 25 | Loss: 0.7129
Epoch 0 | Batch 26 | Loss: 0.6884
Epoch 0 | Batch 27 | Loss: 0.6794
Epoch 0 | Batch 28 | Loss: 0.6780
Epoch 0 | Batch 29 | Loss: 0.7198
Epoch 0 | Batch 30 | Lo

TrainOutput(global_step=1920, training_loss=0.518121999502182, metrics={'train_runtime': 690.4671, 'train_samples_per_second': 44.492, 'train_steps_per_second': 2.781, 'total_flos': 2020692905164800.0, 'train_loss': 0.518121999502182, 'epoch': 3.0})

In [ ]:
eval_results = trainer.evaluate()
print("📊 Final Validation Results:", eval_results)


📊 Final Validation Results: {'eval_loss': 0.6998733282089233, 'eval_accuracy': 0.6386292834890965, 'eval_runtime': 5.3657, 'eval_samples_per_second': 239.298, 'eval_steps_per_second': 7.641, 'epoch': 3.0}


In [ ]:
test_output = trainer.predict(test_dataset)

# Get predicted labels
test_preds = test_output.predictions.argmax(-1)

# If your test set has labels (it does for LIAR), compare them:
from sklearn.metrics import accuracy_score, classification_report

test_accuracy = accuracy_score(test_df['label'], test_preds)
print("🧪 Test Accuracy:", test_accuracy)

# Optional: Detailed report
print(classification_report(test_df['label'], test_preds))


🧪 Test Accuracy: 0.6314127861089187
              precision    recall  f1-score   support

           0       0.63      0.37      0.47       553
           1       0.63      0.83      0.72       714

    accuracy                           0.63      1267
   macro avg       0.63      0.60      0.59      1267
weighted avg       0.63      0.63      0.61      1267



In [3]:
# Upgrade pip first (important!)
!pip install --upgrade pip


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [4]:
# Then install datasets cleanly
!pip install datasets --no-deps


  Using cached datasets-3.5.0-py3-none-any.whl.metadata (19 kB)


In [5]:
# Install transformers and sklearn
!pip install transformers scikit-learn


In [7]:
# 2. Install missing dependency for datasets
!pip install multiprocess


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.5.0 requires xxhash, which is not installed.
datasets 3.5.0 requires dill<0.3.9,>=0.3.0, but you have dill 0.3.9 which is incompatible.
datasets 3.5.0 requires fsspec[http]<=2024.12.0,>=2023.1.0, but you have fsspec 2025.3.2 which is incompatible.
datasets 3.5.0 requires multiprocess<0.70.17, but you have multiprocess 0.70.17 which is incompatible.


In [8]:
# BERT + GPT-2 Tokenizer for Fake News Detection (Google Colab Ready)

!pip install transformers datasets sklearn --quiet

import torch
from transformers import BertForSequenceClassification, GPT2Tokenizer, Trainer, TrainingArguments
from datasets import load_dataset, Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd
import numpy as np
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load your uploaded dataset
train_df = pd.read_csv("train.tsv", sep='\t', header=None)
val_df = pd.read_csv("valid.tsv", sep='\t', header=None)
test_df = pd.read_csv("test.tsv", sep='\t', header=None)


def prepare_binary_labels(df):
    df = df[[1, 2]].rename(columns={1: 'label', 2: 'text'})
    fake_labels = ['pants-fire', 'false', 'barely-true']
    df['label'] = df['label'].apply(lambda x: 0 if x in fake_labels else 1)
    return df

train_df = prepare_binary_labels(train_df)
val_df = prepare_binary_labels(val_df)
test_df = prepare_binary_labels(test_df)

# Combine into HuggingFace dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# Preprocessing function
def clean_text(text):
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z ]", "", text)
    text = text.lower()
    text = " ".join([PorterStemmer().stem(w) for w in text.split() if w not in stopwords.words('english')])
    return text

train_dataset = train_dataset.map(lambda x: {'text': clean_text(x['text'])})
val_dataset = val_dataset.map(lambda x: {'text': clean_text(x['text'])})
test_dataset = test_dataset.map(lambda x: {'text': clean_text(x['text'])})

# Tokenizer: GPT-2
gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token  # required for batching

def tokenize_function(examples):
    return gpt2_tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Model
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2).to(device)

# Evaluation
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# Training
training_args = TrainingArguments(
    output_dir="./results-bert-gpt2",
    evaluation_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=gpt2_tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

# Evaluation
results = trainer.evaluate(tokenized_test)
print("\nTest Results (BERT + GPT-2 Tokenizer):")
print(results)


  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


ModuleNotFoundError: No module named 'xxhash'

In [9]:
!pip install xxhash


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.5.0 requires dill<0.3.9,>=0.3.0, but you have dill 0.3.9 which is incompatible.
datasets 3.5.0 requires fsspec[http]<=2024.12.0,>=2023.1.0, but you have fsspec 2025.3.2 which is incompatible.
datasets 3.5.0 requires multiprocess<0.70.17, but you have multiprocess 0.70.17 which is incompatible.
